In [1]:
import pandas as pd

In [2]:
dataframe = pd.read_csv('watson_healthcare_modified.csv')
dataframe.sample(4)

,EmployeeID,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,...,RelationshipSatisfaction,StandardHours,Shift,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
1447,1665956,38,No,Travel_Rarely,345,Cardiology,10,2,Life Sciences,1,...,3,80,1,10,1,3,10,7,1,9
1468,1720201,45,No,Travel_Rarely,1329,Maternity,2,2,Other,1,...,1,80,2,10,3,3,10,7,3,9
1175,1769167,20,No,Travel_Rarely,1141,Cardiology,2,3,Medical,1,...,1,80,0,2,3,3,2,2,2,2
1567,1065186,49,No,Travel_Rarely,1098,Maternity,4,2,Medical,1,...,3,80,1,23,2,4,1,0,0,0


In [3]:
dataframe.shape

(1676, 35)

In [4]:
dataframe.select_dtypes(include='object')

,Attrition,BusinessTravel,Department,EducationField,Gender,JobRole,MaritalStatus,Over18,OverTime
0,No,Travel_Rarely,Cardiology,Life Sciences,Female,Nurse,Single,Y,Yes
1,No,Travel_Frequently,Maternity,Life Sciences,Male,Other,Married,Y,No
2,Yes,Travel_Rarely,Maternity,Other,Male,Nurse,Single,Y,Yes
3,No,Travel_Frequently,Maternity,Life Sciences,Female,Other,Married,Y,Yes
4,No,Travel_Rarely,Maternity,Medical,Male,Nurse,Married,Y,No
...,...,...,...,...,...,...,...,...,...
1671,Yes,Travel_Rarely,Neurology,Technical Degree,Male,Nurse,Single,Y,Yes
1672,No,Travel_Rarely,Cardiology,Marketing,Female,Nurse,Married,Y,Yes
1673,No,Travel_Rarely,Maternity,Life Sciences,Female,Other,Single,Y,No
1674,No,Travel_Rarely,Neurology,Life Sciences,Female,Therapist,Married,Y,No


In [5]:
dframe = dataframe.copy()
dframe.tail(5)
del dataframe # For original dataset safety

### Train/Test Splitting

In [6]:
y = dframe['Attrition']
X = dframe.drop('Attrition', axis=1)

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
display(X_train.shape)
display(X_test.shape)
display(y_train.shape)
display(y_test.shape)

(1340, 34)

(336, 34)

(1340,)

(336,)

Encoding only the train sets to prevent data leakage

In [10]:
data = X_train.select_dtypes(include='int64')
data.columns

Index(['EmployeeID', 'Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EmployeeCount', 'EnvironmentSatisfaction', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome',
       'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours',
       'Shift', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole',
       'YearsSinceLastPromotion', 'YearsWithCurrManager'],
      dtype='object')

In [11]:
df = X_train.select_dtypes(include='object')
display(df.shape)
df.sample(4)

(1340, 8)

,BusinessTravel,Department,EducationField,Gender,JobRole,MaritalStatus,Over18,OverTime
848,Travel_Rarely,Neurology,Technical Degree,Female,Other,Divorced,Y,No
1638,Travel_Frequently,Cardiology,Marketing,Female,Other,Divorced,Y,No
1209,Travel_Rarely,Maternity,Life Sciences,Female,Other,Married,Y,No
1456,Travel_Rarely,Neurology,Medical,Female,Other,Single,Y,No


In [12]:
(dframe.dtypes).unique()

array([dtype('int64'), dtype('O')], dtype=object)

In [13]:
del dframe # Save memory

#### Dropping Over18 and EducationField 

In [14]:
df.drop(columns=['Over18'], inplace=True)
df.sample(6)
display(df.shape)

(1340, 7)

In [15]:
from sklearn.preprocessing import OneHotEncoder

In [16]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='error').set_output(transform='pandas')

### Encoding Objects to Numerical Values

#### Gender and Department encoding (ohe) and merging

In [17]:
enc_gender = ohe.fit_transform(df[['Gender']])
enc_department = ohe.fit_transform(df[['Department']])

In [18]:
enc_gender.sample(5)

,Gender_Female,Gender_Male
28,1.0,0.0
1389,0.0,1.0
1564,1.0,0.0
349,0.0,1.0
151,0.0,1.0


In [19]:
enc_department.sample(4)

,Department_Cardiology,Department_Maternity,Department_Neurology
863,1.0,0.0,0.0
918,0.0,1.0,0.0
944,1.0,0.0,0.0
1456,0.0,0.0,1.0


In [20]:
df = pd.concat([df, enc_department], axis=1).drop(columns=['Department'])
display(df.shape)
df.sample(5)

(1340, 9)

,BusinessTravel,EducationField,Gender,JobRole,MaritalStatus,OverTime,Department_Cardiology,Department_Maternity,Department_Neurology
57,Travel_Rarely,Medical,Female,Nurse,Married,Yes,0.0,1.0,0.0
347,Travel_Frequently,Medical,Male,Other,Single,No,1.0,0.0,0.0
1309,Travel_Rarely,Human Resources,Male,Other,Married,No,0.0,0.0,1.0
524,Travel_Rarely,Technical Degree,Female,Nurse,Single,No,0.0,1.0,0.0
834,Travel_Frequently,Medical,Female,Administrative,Single,No,0.0,1.0,0.0


In [21]:
df = pd.concat([df, enc_gender], axis=1).drop(columns=['Gender'])
display(df.shape)
df.sample(7)

(1340, 10)

,BusinessTravel,EducationField,JobRole,MaritalStatus,OverTime,Department_Cardiology,Department_Maternity,Department_Neurology,Gender_Female,Gender_Male
1597,Travel_Frequently,Life Sciences,Nurse,Married,No,0.0,0.0,1.0,0.0,1.0
1015,Travel_Rarely,Life Sciences,Nurse,Single,No,0.0,1.0,0.0,0.0,1.0
1045,Travel_Rarely,Other,Nurse,Single,No,1.0,0.0,0.0,0.0,1.0
102,Travel_Frequently,Life Sciences,Nurse,Single,Yes,0.0,1.0,0.0,1.0,0.0
1246,Travel_Rarely,Marketing,Other,Single,No,1.0,0.0,0.0,1.0,0.0
1008,Travel_Rarely,Marketing,Nurse,Single,Yes,1.0,0.0,0.0,1.0,0.0
269,Travel_Rarely,Life Sciences,Nurse,Married,No,0.0,1.0,0.0,0.0,1.0


#### OverTime and Attrition encoding (mapping to 1s and 0s)

In [22]:
# yes = 1 | no = 0 
df['Attrition'] = y_train.map({'Yes': 1.0, 'No': 0.0})  # adding y_train set's 'Attrition' column to df
df['OverTime'] = df['OverTime'].map({'Yes': 1.0, 'No': 0.0})

In [23]:
df.sample(7)

,BusinessTravel,EducationField,JobRole,MaritalStatus,OverTime,Department_Cardiology,Department_Maternity,Department_Neurology,Gender_Female,Gender_Male,Attrition
988,Travel_Rarely,Marketing,Nurse,Married,1.0,1.0,0.0,0.0,0.0,1.0,0.0
110,Travel_Frequently,Medical,Nurse,Single,0.0,0.0,1.0,0.0,1.0,0.0,0.0
982,Travel_Rarely,Medical,Therapist,Married,1.0,0.0,1.0,0.0,0.0,1.0,1.0
825,Non-Travel,Medical,Nurse,Single,0.0,0.0,1.0,0.0,0.0,1.0,1.0
387,Travel_Rarely,Marketing,Nurse,Divorced,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1047,Travel_Frequently,Medical,Nurse,Single,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1006,Travel_Rarely,Medical,Administrative,Married,1.0,0.0,1.0,0.0,1.0,0.0,0.0


#### Ordinal encoding for BusinessTravel and MaritalStatus

In [24]:
travel_frequency = ['Non-Travel', 'Travel_Rarely', 'Travel_Frequently'] # 0.0 = Non-Travel, 1.0 = Travel_Rarely, 2.0 = Travel_Frequently
marital_status = ['Single','Married', 'Divorced']

In [25]:
from sklearn.preprocessing import OrdinalEncoder

In [26]:
oe_travel = OrdinalEncoder(categories=[travel_frequency])
oe_marital_status = OrdinalEncoder(categories= [marital_status])

In [27]:
df['BusinessTravel'] = oe_travel.fit_transform(df[['BusinessTravel']])

In [28]:
df['MaritalStatus'] = oe_marital_status.fit_transform(df[['MaritalStatus']])

In [29]:
df.sample(10)

,BusinessTravel,EducationField,JobRole,MaritalStatus,OverTime,Department_Cardiology,Department_Maternity,Department_Neurology,Gender_Female,Gender_Male,Attrition
459,1.0,Medical,Nurse,2.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
791,2.0,Life Sciences,Nurse,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
986,1.0,Life Sciences,Other,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1327,1.0,Medical,Administrative,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
61,2.0,Life Sciences,Nurse,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
202,2.0,Medical,Other,2.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1178,1.0,Life Sciences,Nurse,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
1477,1.0,Medical,Therapist,2.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
720,2.0,Medical,Other,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1547,2.0,Marketing,Therapist,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [30]:
max_len = max(len(col) for col in df.columns) # finds the length of the largest column name

for col in df.columns:
    print(f'{col:<{max_len}} = {df[col].unique()}') # left aligns the column name

del max_len

BusinessTravel        = [1. 0. 2.]
EducationField        = ['Technical Degree' 'Medical' 'Life Sciences' 'Marketing' 'Other'
 'Human Resources']
JobRole               = ['Other' 'Nurse' 'Therapist' 'Administrative' 'Admin']
MaritalStatus         = [0. 1. 2.]
OverTime              = [0. 1.]
Department_Cardiology = [0. 1.]
Department_Maternity  = [0. 1.]
Department_Neurology  = [1. 0.]
Gender_Female         = [0. 1.]
Gender_Male           = [1. 0.]
Attrition             = [0. 1.]


In [31]:
df[df['JobRole'] == 'Other'].count()

BusinessTravel           428
EducationField           428
JobRole                  428
MaritalStatus            428
OverTime                 428
Department_Cardiology    428
Department_Maternity     428
Department_Neurology     428
Gender_Female            428
Gender_Male              428
Attrition                428
dtype: int64

In [32]:
df[df['EducationField'] == 'Other'].count()

BusinessTravel           74
EducationField           74
JobRole                  74
MaritalStatus            74
OverTime                 74
Department_Cardiology    74
Department_Maternity     74
Department_Neurology     74
Gender_Female            74
Gender_Male              74
Attrition                74
dtype: int64

In [33]:
df.sample(7)

,BusinessTravel,EducationField,JobRole,MaritalStatus,OverTime,Department_Cardiology,Department_Maternity,Department_Neurology,Gender_Female,Gender_Male,Attrition
1138,1.0,Medical,Other,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1287,2.0,Life Sciences,Nurse,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
446,1.0,Marketing,Nurse,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1441,1.0,Life Sciences,Therapist,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
151,1.0,Marketing,Nurse,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
7,1.0,Life Sciences,Nurse,2.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
445,0.0,Life Sciences,Nurse,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


#### JobRole and EducationField encoding (categorical mapping)

In [34]:
df['EducationField'] = df['EducationField'].map({'Other': 0.0, 'Life Sciences': 1.0, 'Medical': 2.0, 'Marketing': 3.0, 'Technical Degree': 4.0, 'Human Resources': 5.0})
df['is_other_JobRole'] = (df['JobRole'] == 'Other').astype(float)
df['JobRole'] = df['JobRole'].map({'Other': 0.0, 'Nurse': 1.0,'Therapist': 2.0, 'Administrative': 3.0, 'Admin': 4.0})

In [35]:
df.columns

Index(['BusinessTravel', 'EducationField', 'JobRole', 'MaritalStatus',
       'OverTime', 'Department_Cardiology', 'Department_Maternity',
       'Department_Neurology', 'Gender_Female', 'Gender_Male', 'Attrition',
       'is_other_JobRole'],
      dtype='object')

In [36]:
new_column_order = ['BusinessTravel', 'EducationField', 'JobRole', 'MaritalStatus',
       'OverTime', 'Department_Cardiology', 'Department_Maternity',
       'Department_Neurology', 'Gender_Female', 'Gender_Male', 'is_other_JobRole',
       'Attrition']
df = df[new_column_order] # shifted the 'Attrition' column to the end for clarity
df.head(7)

,BusinessTravel,EducationField,JobRole,MaritalStatus,OverTime,Department_Cardiology,Department_Maternity,Department_Neurology,Gender_Female,Gender_Male,is_other_JobRole,Attrition
832,1.0,4.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
266,1.0,2.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
148,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
383,1.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
907,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
613,0.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1320,0.0,1.0,0.0,2.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0


In [37]:
display(df.shape)
df.sample(10)

(1340, 12)

,BusinessTravel,EducationField,JobRole,MaritalStatus,OverTime,Department_Cardiology,Department_Maternity,Department_Neurology,Gender_Female,Gender_Male,is_other_JobRole,Attrition
1308,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0
421,1.0,4.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0
1431,1.0,2.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
1402,1.0,2.0,2.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1270,1.0,2.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
587,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
98,1.0,2.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
169,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1177,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1130,1.0,4.0,1.0,2.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [38]:
df.isna().any()

BusinessTravel           False
EducationField           False
JobRole                  False
MaritalStatus            False
OverTime                 False
Department_Cardiology    False
Department_Maternity     False
Department_Neurology     False
Gender_Female            False
Gender_Male              False
is_other_JobRole         False
Attrition                False
dtype: bool

In [39]:
Object_df = df
del df # For Object data safety

### Scaling Numberical Values

In [40]:
print(f'Number of Columns: {len(data.columns)}')
data.columns

Number of Columns: 26


Index(['EmployeeID', 'Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EmployeeCount', 'EnvironmentSatisfaction', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome',
       'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours',
       'Shift', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole',
       'YearsSinceLastPromotion', 'YearsWithCurrManager'],
      dtype='object')

In [41]:
max_len = max(len(col) for col in data.columns) 
for col in data.columns:
    print(f'{col:<{max_len}}: {data[col].nunique()}')

EmployeeID              : 1340
Age                     : 43
DailyRate               : 797
DistanceFromHome        : 29
Education               : 5
EmployeeCount           : 1
EnvironmentSatisfaction : 4
HourlyRate              : 71
JobInvolvement          : 4
JobLevel                : 5
JobSatisfaction         : 4
MonthlyIncome           : 1128
MonthlyRate             : 1181
NumCompaniesWorked      : 10
PercentSalaryHike       : 15
PerformanceRating       : 2
RelationshipSatisfaction: 4
StandardHours           : 1
Shift                   : 4
TotalWorkingYears       : 40
TrainingTimesLastYear   : 7
WorkLifeBalance         : 4
YearsAtCompany          : 37
YearsInCurrentRole      : 19
YearsSinceLastPromotion : 16
YearsWithCurrManager    : 18


In [42]:
data.columns[data.nunique() == 1]

Index(['EmployeeCount', 'StandardHours'], dtype='object')

In [43]:
print(data['EmployeeCount'].unique())
print(data['StandardHours'].unique())

[1]
[80]


In [44]:
data.shape

(1340, 26)

#### Dropping 'EmployeeCount' and 'StandarHours' due to lack of data variation

In [45]:
# dropping EmployeeCount and StandardHours due to lack of variety 
data.drop(['EmployeeCount', 'StandardHours'], axis=1, inplace=True)

In [46]:
data.shape

(1340, 24)

In [47]:
data.sample(4)

,EmployeeID,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,...,PerformanceRating,RelationshipSatisfaction,Shift,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
672,1795081,58,1272,5,3,3,37,2,3,2,...,3,4,1,24,3,3,6,0,0,4
120,1212660,30,1312,23,3,1,96,1,1,3,...,4,3,3,10,2,2,10,7,0,9
516,1767421,38,243,7,4,4,46,2,2,4,...,4,1,0,8,2,3,7,7,0,5
1567,1065186,49,1098,4,2,1,85,2,5,3,...,3,3,1,23,2,4,1,0,0,0


#### Dropping EmployeeID column

In [48]:
data.drop('EmployeeID', axis=1, inplace=True)

In [49]:
data.head(7)

,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,...,PerformanceRating,RelationshipSatisfaction,Shift,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
832,35,528,8,4,3,100,3,1,3,4323,...,3,2,0,6,2,1,5,4,1,4
266,31,1463,23,3,2,64,2,2,4,5582,...,4,2,1,10,2,3,9,0,7,8
148,41,933,9,4,3,94,3,1,1,2238,...,4,4,1,7,2,3,5,0,1,4
383,22,253,11,3,1,43,3,1,2,2244,...,3,4,1,2,1,3,2,1,1,2
907,23,373,1,2,4,47,3,1,3,1223,...,4,4,1,1,2,3,1,0,0,1
613,27,443,3,3,4,50,3,1,4,1706,...,3,3,3,0,6,2,0,0,0,0
1320,28,280,1,2,3,43,3,1,4,2706,...,3,2,1,3,2,3,3,2,2,2


In [50]:
Object_df.head(7)

,BusinessTravel,EducationField,JobRole,MaritalStatus,OverTime,Department_Cardiology,Department_Maternity,Department_Neurology,Gender_Female,Gender_Male,is_other_JobRole,Attrition
832,1.0,4.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
266,1.0,2.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
148,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
383,1.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
907,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
613,0.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1320,0.0,1.0,0.0,2.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0


In [51]:
display(Object_df.shape)
Object_df.columns

(1340, 12)

Index(['BusinessTravel', 'EducationField', 'JobRole', 'MaritalStatus',
       'OverTime', 'Department_Cardiology', 'Department_Maternity',
       'Department_Neurology', 'Gender_Female', 'Gender_Male',
       'is_other_JobRole', 'Attrition'],
      dtype='object')

In [52]:
Object_df.shape

(1340, 12)

In [53]:
data.shape

(1340, 23)

In [54]:
this_df = pd.concat([data, Object_df], axis=1)
display(this_df.shape)
this_df.sample(7)

(1340, 35)

,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,...,JobRole,MaritalStatus,OverTime,Department_Cardiology,Department_Maternity,Department_Neurology,Gender_Female,Gender_Male,is_other_JobRole,Attrition
734,48,1355,4,4,3,78,2,3,3,10999,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
63,59,1435,25,3,1,99,3,3,1,7637,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1495,50,854,1,4,4,68,3,5,4,19517,...,4.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
983,39,1498,21,4,1,44,2,2,4,6120,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
587,29,805,1,2,2,36,2,1,1,2319,...,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
336,29,318,8,4,2,77,1,1,1,2119,...,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
243,40,1300,24,2,1,62,3,2,4,3319,...,0.0,2.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0


In [56]:
this_df.to_csv('preprocessed_data.csv', index=False)